# Assumptions and Inference


## Assumptions
As we discussed briefly last week, there are a number of generic assumptions associated with the GLM framework. Much like the standard linear model, we must assess whether these appear to be met well enough in our model to justify our conclusions. However, the complication of the GLM is that not all these assumptions are relevant and not every metric is interpretable, depending upon the specific choice of model. As such, we must first start by considering *how* these assumptions manifest under binary logistic regression, and whether any of the standard assessments remain relevant or useful.


Under the binary logistic GLM we have been using, the conditional distribution of the outcome variable is

$$
    (y_{i} | x_{i1}, x_{i2}, \dots, x_{ik}) \sim \text{Bern}(\pi_{i}).
$$

In words, the distribution of $y_{i}$, conditional on the values of all the predictor variables, is assumed to be Bernoulli with a specific probability of success $\pi_{i}$. Thus the three aspects of the GLM become

$$
\begin{align*}
    E(y_{i})   &= \pi_{i} \\
    \log\left(\frac{\pi_{i}}{1 - \pi_{i}}\right) &= \beta_{0} + \beta_{1}x_{i1} + \dots + \beta_{k}x_{ik} \\
    \text{Var}(y_{i}) &= \pi_{i}(1 - \pi_{i})
\end{align*}
$$

This then leads to the following interpretations of the assumptions:

1. *The observations are independent of each other (no correlation).*
    - This remains the case for all GLMs, but is a function of the *design* rather than something we check. If each observation comes from a different subject with no repeated measurements, we can assume independence.
2. *The observations follow the assumed conditional distribution of the outcome variable.*
    - This remains the case, however, for binary data the Bernoulli distribution is *always* true and so does not need establishing or checking.
3. *The transformed mean is accurately described using the assumed regression model.*
    - This is a key assumption that will need to be checked. On the scale of the link function, we assume the relationship can still be captured by a straight-line and may need to use non-linear methods if this is not the case.
4. *The conditional variance follows the form for the assumed distribution.*
    - This remain true, but is trivial for the Bernoulli distribution because the variance depends entirely on the value of $\pi_{i}$. As such, if the probability of success is accurately captured then the variance will automatically be correct as well. 

From this we can conclude that the only real element we need to check is the accuracy of the regression model on the scale of the link function. This does include multiple aspects, such as checking whether a straight line is an accurate model and checking for outliers. However, this is fairly trivial in terms of the number of assumptions that need to be met. This is not always true of other GLMs, but is true of binary logistic regression.

### Residual Plots
Last semester, we saw how `R` will produce a standard set of residual plots to assess the assumptions of the normal linear model. This is also true of GLMs. However, under certain models, some of these plots are not very useful. We will see examples of these in the next part of the lesson. For now, it is worth understanding that only *one* of these plots is actually useful under binary logistic regression. The remainder are either misleading or difficult to interpret. This is not wholly surprising because, as established above, the only element we really need to check is the model fit. Everything else can largely be ignored.

### Plots of the Model Fit
In order to assess the model fit, the best approach is visualisation and there are numerous ways this can be achieved. This can include *added variable plots* and *component plus residual plots* however, for GLMs, the approach we will endorse is the use of *effects plots*.

As discussed last semester, effects plots involve plotting predictions from the model by varying certain variables and keeping others fixed at a constant value. For instance, under standard linear regression, we can plot the effect of $x_{1}$ by evaluating the model prediction for different values of $x_{1}$ while keeping $x_{2}$ and $x_{3}$ fixed at their means

$$
    \hat{y}_{i} = \hat{\beta}_{0} + \hat{\beta}_{1}x_{i1} + \hat{\beta}_{2}\bar{x}_{2} + \hat{\beta}_{3}\bar{x}_{3}.
$$

This works in exactly the same fashion for GLMs. For instance, in our binary logistic regression of *presbyopia*, we might also have *visual acuity* and *reading distance* as additional continuous variables. The value in the effects plot for the probability of presbyopia at age 50 would then be based on

$$
    \pi_{\text{age}=50} = \frac{e^{\beta_{0} + (\beta_{1} \times 50) + (\beta_{2} \times \overline{\text{acuity}}) + (\beta_{3} \times \overline{\text{distance}})}}{1 + e^{\beta_{0} + (\beta_{1} \times 50) + (\beta_{2} \times \overline{\text{acuity}}) + (\beta_{3} \times \overline{\text{distance}})}}.
$$

For categorical predictors this becomes more difficult because the average of a dummy variable will lead to a fractional value partway between the categories. Instead, it makes sense to compute the constant value based upon the average of each of the levels of the predictor. For instance, if the model above also included *sex*, we could take the average of *male* and the average of *female* and then created a grand average from the two. In the `effects` package, this grand average is weighted by the number of observations within each category so that, for unbalanced designs, the average is weighted more heavily towards the group with more observations [^foot-1].

For the purpose of assessing model fit, simply viewing the prediction on its own is not enough. We need to be able to see any *lack of fit*. This involves the addition of the model residuals so we can assess whether the predicted shape actually follows the data. The `effects` package takes this further by showing the LOESS smoothing line through the residuals. The further this deviates from the model fit, the more indication there is that there are effects that the model is not capturing. In addition, viewing this on the *probability* scale is not all that useful, because we want to think of our model as a straight line, not an S-shaped curve. As such, for the purpose of understanding how to adjust the model fit, it is better to view the effects on the scale of the *link function*. This means evaluating

$$
    \log\left(\frac{\pi_{\text{age}=50}}{1 - \pi_{\text{age}=50}}\right) = \beta_{0} + (\beta_{1} \times 50) + (\beta_{2} \times \overline{\text{acuity}}) + (\beta_{3} \times \overline{\text{distance}})
$$

and thus drawing the model on the scale of the log-odds. As we will see in the next part of the lesson, the `effects` package has several options for the scale on which the effects plots are drawn.


[^foot-1]: This can be changed to averaging the groups equally using the `fixed.predictors=list(given.values="equal")` option. This may be preferable if the unbalanced of the groups is not a feature of the design but rather a result of random sampling. Similarly, the use of the mean for continuous predictors can be changed to the median using `fixed.predictors=list(typical=median)`. These can be combined into a single additional argument of `fixed.predictors=list(given.values="equal",typical=median)`, which may be preferable in some circumstances.

## Hypothesis Tests
Finally, once we are satisfied with the assumptions and model fit, we can turn to inference. This is typically conducted within the framework of NHST, meaning that all the same caveats and criticisms from last semester still apply. Indeed, there are further considerations when applying these methods to GLMs because: 

1. The $p$-values are *asymptotic*, meaning that our confidence in their accuracy should *decrease* as the sample size gets smaller.
2. Interpreting the parameter estimates requires consideration of the link function and some careful transformations of the values.

This makes inference somewhat harder within GLMs because the $p$-values have greater uncertainty and the meaning of the estimates is fiddlier to discern. 

### Wald Tests
Last semester, we saw that `R`, as well as most other implementations of linear regression, will automatically provide hypothesis tests on the parameter estimates. These take the form of testing the hypothesis that the true parameter value is 0, for both the intercept and all the slopes. While the utility of the test for the intercept will depend upon the context, the slope tests consider a clear null situation where the relationship in the population is *flat*. If the probability of our data under this assumption is very low, we reject the idea that the relationship in the population is truly flat.

For GLMs, we have a very similar situation. However, there are two key differences:

1. The meaning of a null value of 0 will depend upon the scale of the link function.
2. The test statistics are asymptotic $z$-tests with $p$-values from a standard normal distribution, rather than $t$-tests.

Within the context of binary logistic regression, we have to remember that the parameters are on the scale of the *log-odds* and thus the null value must also be interpretable on this scale. As discussed previously, the intercept represents the log-odds of success when all the predictors are 0. If this probability was 50%, this would lead to odds of 

$$
\frac{0.5}{1 - 0.5} = \frac{0.5}{0.5} = 1,
$$

meaning we expect 1 success for every 1 failure. Taking the log gives

$$
\log{1} = 0.
$$

As such, a hypothesis test of $\mathcal{H}_{0}: \beta_{0} = 0$ is testing the null that the probability of success when all predictors are 0 is 50%. This is the meaning of the test on the intercept.

```{tip} Why Not a Null Probability of 0?
You might think that it would make more sense to test against a probability of 0, rather than 50%. Unfortunately, this has two main problems. Firstly, the null tests are generic across all GLMs and so a null value of 0 is the simplest choice, provided it is defined for every link function. This makes implementation much easier compared to deriving specific nulls for every possible link function and thus creating different tests for every possible type of model. The second problem is that you cannot take the log of 0. So while 0 probability would lead to 0 odds, $\log 0 = -\infty$. We cannot calculate a test statistic with a proposed population value of $-\infty$ and so this cannot be tested.
```

For the slope, we need to remember the interpretation that this represents the log of the odds-ratio, reflecting how the odds change for each unit increase in the predictor. If the relationship were a flat line then the odds would not change as $x$ increases. For instance, say the odds were 3 for both $x = 0$ and $x = 1$. This would lead to

$$
\text{OR} = \frac{3}{3} = 1.
$$

Taking the log then gives

$$
\log{1} = 0.
$$

As such, a hypothesis test of $\mathcal{H}_{0}: \beta_{j} = 0$ is testing the null that the probability of success *does not change* as $x$ increases. This is a *flat line* on the scale of the log-odds and is therefore the meaning of the tests on the slopes. Note that this is *not* saying that the probability is 0. It is saying that the probability, whatever it is, does not change. The probability could be 90%, but if it stays at 90% irrespective of the value of the predictor then the null hypothesis is true.

In both cases, there is a distinction between the scale on which the tests are conducted and the most useful scale for interpretation. Converting the estimates back into something more interpretable does not undermine the tests. The tests remain valid across all the scales we work within. As such, it is perfectly fine to conduct the tests on a less interpretable scale, so long as the null value has some reasonable meaning. If we fail to reject the null then there is no need to do any conversions. If we do reject the null, we can then perform some transformations to interpret the effect in context.

### Analysis of Deviance
As well as individual tests on the coefficients, we also have *omnibus* tests that we can perform on *groups* of coefficients. Last semester we saw how these forms of tests are most clearly conceptualised as *model comparisons*, forming the basis for the traditional ANOVA tests. As such, the ANOVA table is best thought of as a record of several model comparisons, each of which focuses on a specific omnibus hypothesis. This allows for us to test all the coefficients associated with all the dummy variables that represent individual experimental factors and their interactions, irrespective of the number of levels.

GLMs work in exactly the same fashion, allowing us to create models in the same spirit as traditional ANOVA models, but assuming a different marginal distribution to the normal distribution. For binary logistic regression, this means we can use an ANOVA to model the results from a factorial experiment with a binary outcome variable. This is more correctly termed an *analysis of deviance*.

As an example, sticking with out current example of *presbyopia*, perhaps we are interested in the effect of different types of existing vision correction on the probability of developing this condition. We therefore have a factor of *current correction* with 3 levels: *no correction*, *glasses* and *contact lenses*. This can be coded using dummy variables in exactly the same fashion as we saw last semester:

$$
\log\left(\frac{\pi_{i}}{1 - \pi_{i}}\right) = \beta_{0} + \left(\beta_{1} \times \text{CORR1}_{i}\right) + \left(\beta_{2} \times \text{CORR2}_{i}\right)
$$

where $\text{CORR1}$ is the dummy variable associated with *glasses* and $\text{CORR2}$ is the dummy variable associated with *contact lenses* e.g.

$$
\text{CORR1}_{i} = \begin{cases}
  1, & \text{if subject $i$ wears glasses},\\
  0, & \text{otherwise}.
\end{cases}
$$

and

$$
\text{CORR2}_{i} = \begin{cases}
  1, & \text{if subject $i$ wears contact lenses},\\
  0, & \text{otherwise}.
\end{cases}
$$

In this configuration, $\beta_{0}$ will be the *log-odds* of presbyopia in someone with no current vision correction, $\beta_{1}$ will be the *odds-ratio* representing the difference between no vision correction and glasses, and $\beta_{2}$ will be the *odds-ratio* representing the difference between no vision correction and contact lenses. Hopefully it is clear that this is effectively a *oneway ANOVA* within the context of binary logistic regression.

In order to develop an omnibus tests of *current correction*, we compare the following null ($\mathcal{M}_{0}$) and full ($\mathcal{M}_{1}$) models: